# Production Inference Pipeline
Loads the `.joblib` model weights generated by `training.ipynb` and runs the full hybrid forecast on incoming data.

**Pipeline:** Temporal Interpolation → IQR + Isolation Forest Anomaly Imputation → Prophet Baseline → LightGBM Residual Correction

## 1. Setup & Model Loading

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import joblib

warnings.filterwarnings('ignore')

MODELS_DIR = '../Models'
OUTPUT_DIR = '../Outputs'

TARGET_COL = 'Demand_MWh'
FEATURE_CANDIDATES = [
    'Day_of_Week', 'Is_Weekend', 'Is_Holiday',
    'Avg_Temp', 'Rainfall',
    'Lag_1', 'Lag_7', 'Lag_30', 'Rolling_7',
]

In [ ]:
# Load serialized models
try:
    prophet_model = joblib.load(os.path.join(MODELS_DIR, 'prophet_model.joblib'))
    lgbm_model = joblib.load(os.path.join(MODELS_DIR, 'lgbm_model.joblib'))
    iso_forest = joblib.load(os.path.join(MODELS_DIR, 'iso_forest.joblib'))
    print('SUCCESS: Model weights loaded.')
except FileNotFoundError:
    raise RuntimeError('Models not found. Run training.ipynb first.')

## 2. Data Ingestion & Pre-processing

In [ ]:
# Load incoming data (in production, replace with live data source)
df_infer = pd.read_csv(os.path.join(OUTPUT_DIR, 'dataset_daily_processed.csv'))
df_infer['Date'] = pd.to_datetime(df_infer['Date'])

# Temporal interpolation for missing values
df_infer.set_index('Date', inplace=True)
df_infer = df_infer.interpolate(method='time')
df_infer.reset_index(inplace=True)

# Resolve available features
features = [c for c in FEATURE_CANDIDATES if c in df_infer.columns]
df_clean = df_infer.dropna(subset=features).copy()
print(f'Loaded {len(df_clean)} rows with features: {features}')

## 3. Anomaly Detection & Imputation

In [ ]:
# Isolation Forest anomaly scoring
if_predictions = iso_forest.predict(df_clean[features])

# IQR anomaly detection on target (only when target column is available)
if TARGET_COL in df_clean.columns:
    q1 = df_clean[TARGET_COL].quantile(0.25)
    q3 = df_clean[TARGET_COL].quantile(0.75)
    iqr_val = q3 - q1
    iqr_anomalies = np.where(
        (df_clean[TARGET_COL] < q1 - 1.5 * iqr_val) | (df_clean[TARGET_COL] > q3 + 1.5 * iqr_val),
        -1, 1,
    )
    is_anomaly = (if_predictions == -1) | (iqr_anomalies == -1)

    # Impute anomalous target values with trailing 7-day clean mean
    for idx in np.where(is_anomaly)[0]:
        start = max(0, idx - 7)
        clean_mask = ~is_anomaly[start:idx]
        clean_window = df_clean[TARGET_COL].iloc[start:idx][clean_mask]
        imputed = clean_window.mean() if len(clean_window) > 0 else df_clean[TARGET_COL].mean()
        df_clean.iloc[idx, df_clean.columns.get_loc(TARGET_COL)] = imputed

    print(f'Imputed {is_anomaly.sum()} anomalies via IQR + Isolation Forest.')
else:
    is_anomaly = if_predictions == -1
    print(f'No target column — flagged {is_anomaly.sum()} anomalies via Isolation Forest only.')

df_clean['Anomaly_Flag'] = np.where(is_anomaly, 'ALERT', 'OK')

## 4. Hybrid Forecast

In [ ]:
# Prophet baseline
df_prophet = df_clean[['Date']].rename(columns={'Date': 'ds'})
prophet_preds = prophet_model.predict(df_prophet)['yhat'].values

# LightGBM micro-corrections
lgbm_preds = lgbm_model.predict(df_clean[features])

# Final hybrid combination
df_clean['Forecast_MWh'] = prophet_preds + lgbm_preds

print('--- INFERENCE PREVIEW ---')
display(df_clean[['Date', 'Forecast_MWh', 'Anomaly_Flag']].tail(10))

## 5. Export Results

In [ ]:
output_path = os.path.join(OUTPUT_DIR, 'inference_results.csv')
df_clean.to_csv(output_path, index=False)
print(f'\nInference complete. Output saved to: {output_path}')